<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%205/Data_Wrangling_I_Feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  feature engineering
### Coderhouse - Data Science
Profe Jorge Ruiz

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = sns.load_dataset('titanic')
df

In [ ]:
df.info()

In [ ]:
df['rango'] = np.nan
df.loc[df['age']<50, 'rango'] = '<50 años'
df.loc[df['age']>=50, 'rango'] = '>=50 años'
df

In [ ]:
def age(x):
    if x<50:
        return '<50 años'
    elif pd.isnull(x):
        return np.nan
    else:
        return '>=50 años'

df['rango_2'] = df["age"].apply(age)
df


In [ ]:
nombres =  ['<50 años', '>=50 años']
df['rango_3'] = pd.cut(df['age'], bins=[0, 50, 90], labels=nombres)
df['rango_3'].value_counts()

In [ ]:
df

In [ ]:
df[['age', 'rango', 'rango_2', "rango_3"]].head(15)

In [ ]:
print("size es un atributo que devuelve el número total de elementos en el DataFrame o Serie, incluyendo los valores NaN ", df["age"].size)
print("el método count() que cuenta el número de elementos no nulos en un DataFrame o una Serie. ",df["age"].count())

In [ ]:
pd.pivot_table(df,
               index=['rango'],
               columns= ['sex'],
               values=['survived'],
               aggfunc="sum")/df["sex"].count()*100

In [ ]:
female_df = df[(df['sex'] == "female")& (~df['survived'].isnull()) & (~df['age'].isnull())]

# Calcular el porcentaje de supervivencia de mujeres en cada rango de edad
result_df = pd.pivot_table(female_df,
                           index='rango',
                           columns='sex',
                           values='survived',
                           aggfunc="sum")/female_df["sex"].size*100
result_df

In [ ]:
female_df.survived.value_counts()

In [ ]:
male_df = df[(df['sex'] == "male")& (~df['survived'].isnull()) & (~df['age'].isnull())]

# Calcular el porcentaje de supervivencia de mujeres en cada rango de edad
result_df = pd.pivot_table(male_df,
                           index='rango',
                           columns='sex',
                           values='survived',
                           aggfunc="sum")/male_df["sex"].size*100
result_df

In [ ]:
male_df.survived.value_counts()

In [ ]:
df.info()

In [ ]:
# por facilidad vamos a quitar los nulos, pero recuerden que esto es más complejo
#df['age'].dropna(inplace=True) no es la manera correcta
df.dropna(subset=['age'],inplace=True)
df['age'].isnull().sum()


# Feature Engineering

### Feature enconding

#### One-hot enconding

Por reemplazo

In [ ]:
df.alive.value_counts()

In [ ]:
df['alive'] = df['alive'].replace({'no':0, 'yes':1})
df['alive'].value_counts()

In [ ]:
df.survived.value_counts()

Usando get dummies para varias categorias

In [ ]:
df.embark_town.value_counts()

In [ ]:
pd.get_dummies(df.embark_town).head(5)

In [ ]:
pd.get_dummies(df.embark_town, prefix = 'embark_town')

#### Label Encoding

In [ ]:
df['class'].value_counts(dropna=False)

In [ ]:
# Importar el método dentro de la libreria
from sklearn.preprocessing import LabelEncoder
# Instanciar el método
le = LabelEncoder()
# Transformar los datos
df['class_le'] = le.fit_transform(df['class'].astype(str))

In [ ]:
df['class_le'].value_counts()

la codificación se realizará en función del orden alfabético de las etiquetas presentes en la columna. Si deseamos asignar valores específicos, debemos utilizar un enfoque diferente, como crear un diccionario de mapeo antes de aplicar el encoding.

In [ ]:
df[['class', 'class_le']]

#### Ordinal Encoding

In [ ]:
# 1. Importar el método
from sklearn.preprocessing import OrdinalEncoder
# 2. Instanciar el método
ord1 = OrdinalEncoder()

# 3. Transformar los datos
df["class_oe"] = ord1.fit_transform(df[["class"]])

In [ ]:
df["class_oe"].value_counts(dropna=False)

In [ ]:
df[['class', 'class_le', "class_oe"]]

Asignación manual a traves de un diccionario

In [ ]:
diccionario = {'First':1, 'Second':2, 'Third':3}

df['class_oe_dict']= df['class'].map(diccionario)

In [ ]:
df[['class_oe_dict']]

### Feature binning

In [ ]:
df.age.hist()

In [ ]:
df.age.describe()

In [ ]:
cortes = [0, 12, 18, 65, 100]
nombres =  ["Niños", 'Adolescentes', 'Adultos', "Adultos mayores"]

# Agregar la columna 'age_ie' con las etiquetas
df['age_ie'] = pd.cut(df['age'], bins=cortes, labels=nombres)

# Contar los valores de 'age_ie'
value_counts = df['age_ie'].value_counts().sort_values()


value_counts

In [ ]:
df[['age', 'age_ie']]

In [ ]:
df[['age','age_ie']].dtypes

### Transformaciones

### Escalado

**Escalado estándar:** quita la media y divide por la desviación estándar, sensible a outliers <br><br>
$z = \frac{(x - u)}{s}$

In [ ]:
# 1.
from sklearn.preprocessing import StandardScaler
# 2.
ss = StandardScaler()
# 3.
df[['age_ss']] = ss.fit_transform(df[['age']])

In [ ]:
df[['age', 'age_ss']].describe().round(1)

In [ ]:
df[['age', 'age_ss']].hist(figsize=(10,4))

In [ ]:
df[['fare_ss']] = ss.fit_transform(df[['fare']])

df[['fare', 'fare_ss']].describe().round(1)


In [ ]:

df[['fare', 'fare_ss']].hist(figsize=(10,4))

Es importante comprender que la estandarización en Z-score no garantiza que los datos sigan una distribución normal. Aunque los nuevos datos estandarizados tendrán una media de cero y una desviación estándar de uno, la forma de la distribución original puede no cambiar o hacerlo levemente...

 La estandarización es simplemente una transformación de los datos para que tengan una media de cero y una desviación estándar de uno y nos ayude a realizar comparaciones de variables de distinta naturaleza, muy usado como fase previa en algunos modelos de ML.

**MinMax:** Escala los datos en un rango entre 0 y 1, sensible a outliers <br><br>
$X_{std} = \frac{X - X_{min}}{X_{max} - X_{min}}$ <br><br>
$X_{scaled} = X_{std} * (max - min) + min $

In [ ]:
# 1.
from sklearn.preprocessing import MinMaxScaler
# 2.
mm = MinMaxScaler()
# 3.
df[['age_mm']] = mm.fit_transform(df[['age']])

In [ ]:
df[['age', 'age_mm']].describe()

In [ ]:
df[['age', 'age_mm']].hist(figsize=(10,4))

**Robust scaler:** quita la mediana y escala de acuerdo al rango intercuartílico <br><br>

$ X_{scaled} = \frac{X_i - X_{med}}{X_{q75} - X_{q25}}$

In [ ]:
# 1.
from sklearn.preprocessing import RobustScaler
# 2.
rs = RobustScaler()
# 3.
df[['age_rs', 'fare_rs']] = rs.fit_transform(df[['age', 'fare']])

In [ ]:
df[['age', 'age_rs']].describe()

In [ ]:
df[['age', 'age_rs']].hist(figsize=(10,4))